In [2]:
# Load data to the dataframe as a starting point to create the gold layer

df = spark.read.table("saleshouse.sales_silver")

display(df.limit(5))

StatementMeta(, da34f171-bd52-4e0c-b474-50df79e7dea1, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f6876c2d-f31d-4fd7-a5fe-6f824be00f13)

In [4]:
# create date dimension table

from pyspark.sql.types import *
from delta.tables import *

# Define the schema for the dimdate_gold table

DeltaTable.createIfNotExists(spark) \
.tableName("saleshouse.dimdate_gold") \
.addColumn("OrderDate", DateType()) \
.addColumn("Day", IntegerType()) \
.addColumn("Month", IntegerType()) \
.addColumn("Year", IntegerType()) \
.addColumn("mmmyyyy", StringType()) \
.addColumn("yyyymmm", StringType()) \
.execute()



StatementMeta(, b9e04e9a-6f4f-470e-9ab9-6f180a767275, 6, Finished, Available, Finished, False)

NameError: name 'dfdimdate_gold' is not defined

In [7]:
# Create dataframe for date dimension, dimdate_gold

from pyspark.sql.functions import col, dayofmonth, month, year, date_format

dfdimdate_gold = df.drop_duplicates(["OrderDate"]).select(col("OrderDate"), \
        dayofmonth("OrderDate").alias("Day"),\
        month("OrderDate").alias("Month"), \
        year("OrderDate").alias("Year"), \
        date_format(col("OrderDate"), "MMM-yyyy").alias("mmmyyyy"), \
        date_format(col("OrderDate"), "yyyy-MMM").alias("yyyymmm"), \
    ).orderBy("OrderDate")

# Display the first 10 rows of the dataframe to preview your data

display(dfdimdate_gold.limit(10))

StatementMeta(, b9e04e9a-6f4f-470e-9ab9-6f180a767275, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 35cc9d46-eb60-4939-98fe-126739205114)

In [8]:
# update the date dimension as new data comes in

from delta.tables import *

deltaTable = DeltaTable.forPath(spark, "Tables/dimdate_gold")

dfUpdates = dfdimdate_gold

deltaTable.alias("gold")\
    .merge(
        dfUpdates.alias("updates"),
        "gold.OrderDate = updates.OrderDate"
    )\
    .whenMatchedUpdate(set =
    {

    })\
    .whenNotMatchedInsert(values=
    {
        "OrderDate": "updates.OrderDate",
        "Day": "updates.Day",
        "Month":"updates.Month",
        "Year": "updates.Year",
        "mmmyyyy":"updates.mmmyyyy",
        "yyyymmm": "updates.yyyymmm"

    }
    )\
    .execute()

StatementMeta(, b9e04e9a-6f4f-470e-9ab9-6f180a767275, 10, Finished, Available, Finished, False)

In [9]:
# customer dimension table

from pyspark.sql.types import *
from delta.tables import *

# Create customer_gold dimension delta table

DeltaTable.createIfNotExists(spark)\
.tableName("saleshouse.dimcustomer_gold")\
.addColumn("CustomerName", StringType()) \
.addColumn("Email", StringType()) \
.addColumn("First", StringType()) \
.addColumn("Last", StringType()) \
.addColumn("CustomerID", LongType()) \
.execute()

StatementMeta(, b9e04e9a-6f4f-470e-9ab9-6f180a767275, 11, Finished, Available, Finished, False)

In [3]:
# drop duplicate customers, select specific columns, and split the “CustomerName” column to create “First” and “Last” name columns

from pyspark.sql.functions import col, split

dfdimCustomer_silver = df.dropDuplicates(["CustomerName","Email"]).select(col("CustomerName"), col("Email")) \
.withColumn("First",split(col("CustomerName"), " ").getItem(0)) \
.withColumn("Last",split(col("CustomerName"), " ").getItem(1))

# Display the first 10 rows of the dataframe to preview your data

display(dfdimCustomer_silver.limit(10))


StatementMeta(, da34f171-bd52-4e0c-b474-50df79e7dea1, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 562e5211-bec7-4a69-bee5-acfd31ebe4d6)

In [5]:
# create the ID column for our customers
from pyspark.sql.functions import monotonically_increasing_id, col, when, coalesce,max, lit

dfdimCustomer_temp = spark.read.table("SalesHouse.dimcustomer_gold")

MAXCustomerID = dfdimCustomer_temp.select(coalesce(max(col("CustomerID")),lit(0)).alias("MAXCustomerID")).first()[0]

dfdimCustomer_gold = dfdimCustomer_silver.join(dfdimCustomer_temp, (dfdimCustomer_silver.CustomerName == dfdimCustomer_temp.CustomerName) \
& (dfdimCustomer_silver.Email == dfdimCustomer_temp.Email), "left_anti")

dfdimCustomer_gold = dfdimCustomer_gold.withColumn("CustomerID", monotonically_increasing_id() + MAXCustomerID + 1)

# Display the first 10 rows of the dataframe to preview your data

display(dfdimCustomer_gold.limit(10))

'''cleaning and transforming customer data (dfdimCustomer_silver) by 
performing a left anti join to exclude duplicates that already exist in the dimCustomer_gold table, 
and then generating unique CustomerID values using the monotonically_increasing_id() function.'''builtin


StatementMeta(, da34f171-bd52-4e0c-b474-50df79e7dea1, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 377b2170-4210-428f-9c54-500d0bf4b764)

In [6]:
# ensure that your customer table remains up-to-date as new data comes

from delta.tables import *

deltaTable = DeltaTable.forPath(spark, "Tables/dimcustomer_gold")

dfUpdates = dfdimCustomer_gold

deltaTable.alias("gold")\
.merge(
    dfUpdates.alias("updates"), 
    "gold.CustomerName = updates.CustomerName AND gold.Email = updates.Email"
)\
.whenMatchedUpdate(set=
{

})\
.whenNotMatchedInsert(values=
{
    "CustomerName": "updates.CustomerName",
    "Email": "updates.Email",
    "First":"updates.First",
    "Last":"updates.Last",
    "CustomerID": "updates.CustomerID"
})\
.execute()

StatementMeta(, da34f171-bd52-4e0c-b474-50df79e7dea1, 8, Finished, Available, Finished, False)

In [7]:
# product dimension

from pyspark.sql.types import *
from delta.tables import *

DeltaTable.createIfNotExists(spark) \
.tableName("SalesHouse.dimproduct_gold") \
.addColumn("ItemName", StringType()) \
.addColumn("ItemID", LongType()) \
.addColumn("ItemInfo", StringType()) \
.execute()

StatementMeta(, da34f171-bd52-4e0c-b474-50df79e7dea1, 9, Finished, Available, Finished, False)

In [8]:
# product_silver dataframe.

from pyspark.sql.functions import col, split, lit, when

# Create product_silver dataframe

dfdimProduct_silver = df.dropDuplicates(["Item"]).select(col("Item")) \
.withColumn("ItemName", split(col("Item"), ",").getItem(0)) \
.withColumn("ItemInfo", when((split(col("Item"), ",").getItem(1).isNull() \
| (split(col("Item"), ",").getItem(1)=="")),lit("")) \
.otherwise(split(col("Item"), ",").getItem(1)))

# Display the first 10 rows of the dataframe to preview your data
display(dfdimProduct_silver.limit(10))

StatementMeta(, da34f171-bd52-4e0c-b474-50df79e7dea1, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 88103513-9f07-4cf4-94e2-0b8a22460b01)

In [9]:
# create IDs for dimProduct_gold table.

from pyspark.sql.functions import monotonically_increasing_id, col, lit, max, coalesce

# dfdimProduct_temp = dfdimProduct_silver

dfdimProduct_temp = spark.read.table("SalesHouse.dimproduct_gold")

MAXProductID = dfdimProduct_temp.select(coalesce(max(col("ItemID")),lit(0)).alias("MAXItemID")).first()[0]

dfdimProduct_gold = dfdimProduct_silver.join(dfdimProduct_temp, (dfdimProduct_silver.ItemName == dfdimProduct_temp.ItemName ) \
 & (dfdimProduct_silver.ItemInfo == dfdimProduct_temp.ItemInfo), "left_anti")

dfdimProduct_gold = dfdimProduct_gold.withColumn("ItemID", monotonically_increasing_id() + MAXProductID + 1)

# Display the first 10 rows of the dataframe to preview your data

display(dfdimProduct_gold.limit(10))

'''This calculates the next available product ID based on the current data in the table, 
assigns these new IDs to the products, and then displays the updated product information.'''

StatementMeta(, da34f171-bd52-4e0c-b474-50df79e7dea1, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6ed36790-de44-4324-ac54-f0b624d88e4e)

In [10]:
# ensure that your product table remains up-to-date as new data comes in

from delta.tables import *

deltaTable = DeltaTable.forPath(spark, "Tables/dimproduct_gold")

dfUpdates = dfdimProduct_gold

deltaTable.alias("gold") \
.merge(
    dfUpdates.alias("updates"),
    "gold.ItemName = updates.ItemName AND gold.ItemInfo = updates.ItemInfo"
)\
.whenMatchedUpdate(set = 
{

}
)\
.whenNotMatchedInsert(values=
{
    "ItemName":"updates.ItemName",
    "ItemInfo": "updates.ItemInfo",
    "ItemID":"updates.ItemID"
})\
.execute()



StatementMeta(, da34f171-bd52-4e0c-b474-50df79e7dea1, 12, Finished, Available, Finished, False)

In [22]:
# create the fact table

from pyspark.sql.types import *
from delta.tables import *

DeltaTable.createIfNotExists(spark)\
.tableName("saleshouse.factsales_gold") \
.addColumn("CustomerID", LongType()) \
.addColumn("ItemID", LongType()) \
.addColumn("OrderDate", DateType()) \
.addColumn("Quantity", IntegerType()) \
.addColumn("UnitPrice", FloatType()) \
.addColumn("Tax", FloatType()) \
.execute()

StatementMeta(, da34f171-bd52-4e0c-b474-50df79e7dea1, 24, Finished, Available, Finished, False)

In [23]:
'''create a new dataframe to combine sales data with customer and product information 
include customer ID, item ID, order date, quantity, unit price, and tax'''

from pyspark.sql.functions import col
    
dfdimCustomer_temp = spark.read.table("SalesHouse.dimcustomer_gold")
dfdimProduct_temp = spark.read.table("SalesHouse.dimproduct_gold")
    
df = df.withColumn("ItemName",split(col("Item"), ", ").getItem(0)) \
    .withColumn("ItemInfo",when((split(col("Item"), ", ").getItem(1).isNull() | (split(col("Item"), ", ").getItem(1)=="")),lit("")).otherwise(split(col("Item"), ", ").getItem(1))) \
    
    
# Create Sales_gold dataframe
    
dffactSales_gold = df.alias("df1").join(dfdimCustomer_temp.alias("df2"), \
(df.CustomerName == dfdimCustomer_temp.CustomerName) & (df.Email == dfdimCustomer_temp.Email), "left") \
        .join(dfdimProduct_temp.alias("df3"),(df.ItemName == dfdimProduct_temp.ItemName) & \
        (df.ItemInfo == dfdimProduct_temp.ItemInfo), "left") \
    .select(col("df2.CustomerID") \
        , col("df3.ItemID") \
        , col("df1.OrderDate") \
        , col("df1.Quantity") \
        , col("df1.UnitPrice") \
        , col("df1.Tax") \
    ).orderBy(col("df1.OrderDate"), col("df2.CustomerID"), col("df3.ItemID"))
    
# Display the first 10 rows of the dataframe to preview your data
    
display(dffactSales_gold.head(10))


StatementMeta(, da34f171-bd52-4e0c-b474-50df79e7dea1, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 32806076-48a7-4cde-a7e3-49fadd2222fe)

In [24]:
# ensure that sales data remains up-to-date

from delta.tables import *

deltaTable = DeltaTable.forPath(spark, "Tables/factsales_gold")

dfUpdates = dffactSales_gold

deltaTable.alias("gold") \
.merge(
    dfUpdates.alias("updates"),
    "gold.OrderDate = updates.OrderDate AND \
    gold.CustomerID = updates.CustomerID AND \
    gold.ItemID = updates.ItemID"
)\
.whenMatchedUpdate(set =
{

}
) \
.whenNotMatchedInsert(values=
{
    "CustomerID":"updates.CustomerID",
    "ItemID":"updates.ItemID",
    "OrderDate":"updates.OrderDate",
    "Quantity":"updates.Quantity",
    "UnitPrice":"updates.UnitPrice",
    "Tax": "updates.Tax"
    
}
)\
.execute()

'''using Delta Lake’s merge operation to synchronize and update the factsales_gold table 
with new sales data (dffactSales_gold). The operation compares the order date, customer ID, and 
item ID between the existing data (silver table) and the new data (updates DataFrame), 
updating matching records and inserting new records as needed.'''
 


StatementMeta(, da34f171-bd52-4e0c-b474-50df79e7dea1, 26, Finished, Available, Finished, False)